#Intro

Dans ce notebook :

- Lecture de Dataframe à partir de fichier
- Ecriture de Dataframe à partir de fichier
- Lecture de Dataframe à partir d'une table
- Ecrture d'un dataframe dans une table

## Fichiers et Dataframes
### Lecture d'un dataframe à partir d'un fichier

Dans un premier temps il faut upload le fichier dans un volumes accessible.

Dans Worksapce cliquer sur Resources et clique droit sur test.csv pour télécharger le fichier.

![](./Resources/up1.png)

Puis aller dans l'onglet catalog et cliquer sur le + puis "Upload to Volume", créer un dossier Cours3 dans le volume et déposé le fichier test.csv à l'intérieur.

![](./Resources/up2.png)

In [0]:

# Lecture d'un fichier CSV avec header et inférence des types
df_from_vol = spark.read.option("sep", ";").option("header", True).option("inferSchema", True).csv("/Volumes/workspace/databricks_training/training/Cours3/test.csv")

display(df_from_vol)  # Affiche le DataFrame dans le notebook Databricks

### Ecriture d'un Dataframe dans un volume

In [0]:
df_simple = spark.createDataFrame([
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie")
], ["id", "name"])


# 1) Dossier temporaire dans le volume
tmp_dir = "/Volumes/workspace/databricks_training/training/Cours3/tmp_write_csv"

# 2) Écriture en une seule partition + header
(
  df_simple.coalesce(1)
           .write.mode("overwrite")
           .option("header", True)
           .csv(tmp_dir)
)

# 3) Trouver le fichier CSV (part-*.csv) et le copier/renommer
files = dbutils.fs.ls(tmp_dir)
part_csv = [f.path for f in files if f.name.endswith(".csv")][0]

target = "/Volumes/workspace/databricks_training/training/Cours3/test_write.csv"
dbutils.fs.cp(part_csv, target)        # copie vers le nom final
dbutils.fs.rm(tmp_dir, recurse=True)   # nettoyage du dossier temporaire


## Tables et Dataframes
### Lecture d'un dataframe à partir du Catalog / Table

Par defaut lors de la création d'un workspace databricks des tables sont présentes dans un catalog de test.

Dans ce cas on vas utiliser le catalog samples et plus spécifiquement : 
samples.nyctaxi.trips


In [0]:

# Spécifie le catalog, le schema et la table
df = spark.table("samples.nyctaxi.trips")

# Ou avec spark.read.table
df = spark.read.table("samples.nyctaxi.trips")

display(df)  # Affiche le DataFrame dans le notebook


On peut également manipuler les tables avec cellules SQL

In [0]:
%sql
SELECT * FROM samples.nyctaxi.trips

## Ecriture d'un dataframe dans une table de catalog
L'objectif est donc d'écrire la table trips dans le catalog que l'on à créer : adbi-training.training

In [0]:
# Écriture du DataFrame dans la table du catalog training.training.trips
df.write.mode("overwrite").saveAsTable("workspace.databricks_training.trips")

In [0]:
# vérifions que la table à bien été écrite
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA databricks_training")

df_test = spark.table("trips")
display(df_test)

## Création de vue temporaires
Dans certains il peut être plus simple de transformer un dataframe en table temporaire.

####DataFrame vs Table Temporaire
DataFrame

- Plus rapide pour les calculs en mémoire (PySpark API).
- Pas de surcharge liée à la création d’une vue SQL.
- Idéal pour transformations programmatiques et gros volumes.
- Moins coûteux car pas d’opération supplémentaire (juste des transformations Spark).

Table Temporaire (Temp View)

- Crée une vue logique sur le DataFrame.
-Permet d’exécuter des requêtes SQL (interopérabilité).
- Pas de copie des données → coût faible, mais chaque requête SQL est traduite en plan Spark.
- Légèrement plus lent si tu fais beaucoup d’aller-retour entre SQL et PySpark (car conversion logique).
- Une temp view ne crée pas de table physique (Delta, Parquet).
- Idéal pour des calculs temporaires ou des POC sans polluer le catalog.



In [0]:
# transformons df_test en vue temporaire
df_test.createOrReplaceTempView("trips_temp")


# Tu peux maintenant l'utiliser en SQL dans le notebook :
display(spark.sql("SELECT * FROM trips_temp"))

In [0]:
%sql
-- et maintenant en SQL
SELECT * FROM trips_temp;

In [0]:
%sql
-- et si on supprime toutes les lignes qui ont pour valeur 10110 dans la colonne dropoff_zip 
select count(*) as nb_trips, (select count(*) FROM trips_temp WHERE dropoff_zip = 10110)  as nb_trips_10110 from trips_temp;

In [0]:
%sql
-- avec les resultats de la requête précédente après la suppréssion on devrait avoir 21158 lignes
DELETE FROM trips_temp WHERE dropoff_zip = 10110

In [0]:
# vérifions avec Python / Spark le dataframe modifié
display(spark.sql("SELECT COUNT(*) FROM trips_temp"))